# `TpDoc` — resolução de tipos de documento do CSK-DFE

Este notebook demonstra a classe `TpDoc`, que resolve um tipo de documento fiscal a partir do nome, do código de domínio (0 a 63) ou do código reverso — o valor efetivamente gravado no campo de 7 bits da chave.

É documentação executável, sem asserções: a verificação do comportamento é responsabilidade da suíte `pytest` em `tests/`.

In [1]:
from csk_dfe import TpDoc, CodigoForaDaFaixaError, TabelaEstendidaError, NomeInexistenteError

## Resolução por código, por código reverso e por nome

In [2]:
TpDoc.from_cod(5)

TpDoc(codigo=5, reverso=80, nome='NFCe')

In [3]:
TpDoc.from_reverse_cod(80)

TpDoc(codigo=5, reverso=80, nome='NFCe')

In [4]:
TpDoc.from_name("MDFe")

TpDoc(codigo=15, reverso=120, nome='MDFe')

## A tabela dos 64 códigos

Códigos reservados (sem `Tipo` no CSV de origem) aparecem marcados — são válidos para `from_cod()` e `from_reverse_cod()`, mas não são resolvíveis por `from_name()`.

In [5]:
print(f"{'código':>6} {'reverso':>7}  nome")
for codigo in range(64):
    tipo = TpDoc.from_cod(codigo)
    nome = tipo.get_name() if tipo.get_name() else "(reservado)"
    print(f"{tipo.get_cod():>6} {tipo.get_reverse_cod():>7}  {nome}")

código reverso  nome
     0       0  NFe
     1      64  NFe Evento
     2      32  (reservado)
     3      96  (reservado)
     4      16  (reservado)
     5      80  NFCe
     6      48  NFCe Evento
     7     112  (reservado)
     8       8  (reservado)
     9      72  (reservado)
    10      40  CTe
    11     104  CTe Evento
    12      24  (reservado)
    13      88  (reservado)
    14      56  (reservado)
    15     120  MDFe
    16       4  MDFe Evento
    17      68  (reservado)
    18      36  (reservado)
    19     100  (reservado)
    20      20  (reservado)
    21      84  (reservado)
    22      52  (reservado)
    23     116  (reservado)
    24      12  (reservado)
    25      76  (reservado)
    26      44  (reservado)
    27     108  (reservado)
    28      28  (reservado)
    29      92  (reservado)
    30      60  (reservado)
    31     124  (reservado)
    32       2  EFD
    33      66  (reservado)
    34      34  (reservado)
    35      98  (reservado)
    36     

## A reversão de 7 bits

O campo de documento da chave grava o código **reverso**, nunca o direto. O bit mais à direita do reverso é sempre `0` para a tabela base — é o sinalizador de tabela estendida.

In [6]:
for codigo in (0, 1, 5, 16, 32, 63):
    tipo = TpDoc.from_cod(codigo)
    print(f"{codigo:>2} = {codigo:07b}  ->  {tipo.get_reverse_cod():>3} = {tipo.get_reverse_cod():07b}")

 0 = 0000000  ->    0 = 0000000
 1 = 0000001  ->   64 = 1000000
 5 = 0000101  ->   80 = 1010000
16 = 0010000  ->    4 = 0000100
32 = 0100000  ->    2 = 0000010
63 = 0111111  ->  126 = 1111110


## Erros de domínio

Entradas inválidas levantam exceções distinguíveis, todas derivadas de `TpDocError` (por sua vez, de `ValueError`).

In [7]:
try:
    TpDoc.from_cod(64)
except CodigoForaDaFaixaError as erro:
    print(f"CodigoForaDaFaixaError: {erro}")

CodigoForaDaFaixaError: código 64 fora da faixa 0 a 63


In [8]:
try:
    TpDoc.from_reverse_cod(81)
except TabelaEstendidaError as erro:
    print(f"TabelaEstendidaError: {erro}")

TabelaEstendidaError: código reverso 81 é ímpar: sinaliza tabela estendida, fora de escopo


In [9]:
try:
    TpDoc.from_name("documento-que-nao-existe")
except NomeInexistenteError as erro:
    print(f"NomeInexistenteError: {erro}")

NomeInexistenteError: nome 'documento-que-nao-existe' não consta da tabela de tipos de documento
